# MLB Wins Prediction — Clean Submission Pipeline

**Best public score: 2.98765** (50-50 average of QR + ElasticNet, no franchise features)

Pipeline: Load → Engineer → Scale → ElasticNet + QR → Average → Submit

In [55]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet, QuantileRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path(os.environ.get('LOCAL_DATA_DIR', Path.cwd())).resolve() / 'data'
SUBMISSIONS_DIR = Path.cwd().resolve() / 'submissions'
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
COMPETITION_NAME = 'sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics'
print(f'Data dir: {DATA_DIR}')
print(f'Submissions dir: {SUBMISSIONS_DIR}')

Data dir: /home/fredc/NTU-DSAI/kaggle-ha/data
Submissions dir: /home/fredc/NTU-DSAI/kaggle-ha/submissions


## 1. Load Data

- **Train**: 1,812 team-seasons (1904-2016), 51 columns including target `W`
- **Predict**: 453 team-seasons, no `W` column

In [57]:
# Use key-enriched files so external joins (e.g., Lahman) can be done safely by year/team
data_df = pd.read_csv(DATA_DIR / 'data_year_team_franchise.csv')
predict_df = pd.read_csv(DATA_DIR / 'predict_year_team_franchise.csv')
print(f'Train: {data_df.shape}  |  Predict: {predict_df.shape}')

Train: (1812, 52)  |  Predict: (453, 48)


## 2. Feature Engineering

24 domain-driven features computed per team-season:
- **Run-based**: `run_diff`, `run_diff_pg`, Pythagorean win expectancy (fixed 1.83 exponent),
  PythagenPat (dynamic exponent), league-relative R and RA per game
- **Offensive rates**: batting average, OBP proxy, SLG proxy (with explicit singles),
  OPS, ISO (isolated power), HR/AB, BB rate, SO rate
- **Pitching/defense**: WHIP, K/BB ratio, HR rate allowed, FIP proxy, ERA vs league average
- **Usage rates**: save rate, complete game rate, shutout rate per game

In [58]:
def add_engineered_features(df):
    d = df.copy()
    G  = d['G']
    IP = d['IPouts'] / 3
    PA = d['AB'] + d['BB']

    d['run_diff']        = d['R'] - d['RA']
    d['run_diff_pg']     = d['run_diff'] / G
    d['pythag_wp']       = d['R']**1.83 / (d['R']**1.83 + d['RA']**1.83 + 1e-9)
    d['pyth_wins']       = d['pythag_wp'] * G
    dyn_exp = ((d['R'] + d['RA']) / (G + 1e-9)) ** 0.287
    d['pythagenport_wp'] = d['R']**dyn_exp / (d['R']**dyn_exp + d['RA']**dyn_exp + 1e-9)
    d['pythagenport_wins'] = d['pythagenport_wp'] * G

    d['batting_avg']     = d['H'] / d['AB']
    d['obp_proxy']       = (d['H'] + d['BB']) / PA
    singles              = d['H'] - d['2B'] - d['3B'] - d['HR']
    d['slg_proxy']       = (singles + 2*d['2B'] + 3*d['3B'] + 4*d['HR']) / (d['AB'] + 1e-9)
    d['ops_proxy']       = d['obp_proxy'] + d['slg_proxy']
    d['iso']             = d['slg_proxy'] - d['batting_avg']
    d['hr_rate_off']     = d['HR'] / d['AB']
    d['bb_rate']         = d['BB'] / PA
    d['so_rate']         = d['SO'] / PA

    d['whip']            = (d['BBA'] + d['HA']) / IP
    d['k_bb_ratio']      = d['SOA'] / d['BBA'].replace(0, float('nan'))
    d['hr_rate_def']     = d['HRA'] / IP
    d['fip_proxy']       = (13*d['HRA'] + 3*d['BBA'] - 2*d['SOA']) / IP + 3.2
    d['era_vs_league']   = d['ERA'] / d['mlb_rpg'].replace(0, float('nan'))

    d['sv_rate']         = d['SV'] / G
    d['cg_rate']         = d['CG'] / G
    d['sho_rate']        = d['SHO'] / G
    d['r_vs_lg']         = d['R'] / G - d['mlb_rpg']
    d['ra_vs_lg']        = d['RA'] / G - d['mlb_rpg']

    return d.fillna(0)


# Apply engineering
data_df    = add_engineered_features(data_df)
predict_df = add_engineered_features(predict_df)

eng_cols = [
    'run_diff','run_diff_pg',
    'pythag_wp','pyth_wins','pythagenport_wp','pythagenport_wins',
    'batting_avg','obp_proxy','slg_proxy','ops_proxy','iso',
    'hr_rate_off','bb_rate','so_rate',
    'whip','k_bb_ratio','hr_rate_def','fip_proxy','era_vs_league',
    'sv_rate','cg_rate','sho_rate','r_vs_lg','ra_vs_lg'
]

print(f'Engineered: {len(eng_cols)}')

Engineered: 24


## 3. Feature Selection & Scaling

**Feature list** — 2 groups combined into `available_features`:
1. **25 base stats** from the dataset (G, R, AB, H, 2B, ... FP, mlb_rpg)
2. **19 one-hot indicators** for era (era_1–era_8) and decade (decade_1910–decade_2010)
3. **24 engineered** features

Only features present in both `data_df` and `predict_df` are kept (safety filter).

**Train/test split**: 80/20 with `random_state=42` for holdout validation.

**Scaling**: `StandardScaler` (zero-mean, unit-variance) applied to continuous features only.
Binary one-hot columns (`era_*`, `decade_*`) are left unscaled since they are already 0/1.

In [59]:
base_features = [
    'G', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB',
    'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA',
    'E', 'DP', 'FP', 'mlb_rpg',
    'era_1', 'era_2', 'era_3', 'era_4', 'era_5', 'era_6', 'era_7', 'era_8',
    'decade_1910', 'decade_1920', 'decade_1930', 'decade_1940', 'decade_1950',
    'decade_1960', 'decade_1970', 'decade_1980', 'decade_1990', 'decade_2000', 'decade_2010',
] + eng_cols

available_features = [c for c in base_features if c in data_df.columns and c in predict_df.columns]
print(f'Total features: {len(available_features)}')

X = data_df[available_features]
y = data_df['W']

# 80/20 split for validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

# Scale continuous features only (leave era_/decade_ binary cols unscaled)
one_hot_cols = [col for col in X_train.columns if col.startswith(('era_', 'decade_'))]
other_cols   = [col for col in X_train.columns if col not in one_hot_cols]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[other_cols] = scaler.fit_transform(X_train[other_cols])
X_test_scaled[other_cols]  = scaler.transform(X_test[other_cols])

X_all_scaled = X.copy()
X_all_scaled[other_cols] = scaler.transform(X[other_cols])

print(f'Scaling {len(other_cols)} continuous + {len(one_hot_cols)} binary features')

Total features: 68
Train: 1449  |  Test: 363
Scaling 48 continuous + 20 binary features


## 4. Model A — ElasticNet (feature selection + prediction)

ElasticNet combines L1 (Lasso) and L2 (Ridge) regularisation. The L1 component zeros out
irrelevant features, performing embedded feature selection; the L2 component stabilises
correlated features.

**Two-pass approach**:
1. **Coarse grid** (`ElasticNetCV`): searches 10 `l1_ratio` values x 60 `alpha` values
   via 5-fold CV. The non-zero coefficients define `active_features` — the subset of
   features that ElasticNet deems informative. This subset is reused by QR in Section 5.
2. **Fine grid**: zooms into a dense neighbourhood around the coarse winner
   (10 `l1_ratio` in 0.90–1.0, 90 `alpha` values around the optimum) for marginal gains.

In [60]:
# --- ElasticNetCV: coarse grid to find active features ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)

enet_cv = ElasticNetCV(
    l1_ratio=[0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99],
    alphas=np.logspace(-4, 2, 60),
    cv=kf, max_iter=50000, tol=1e-4, n_jobs=-1,
)
enet_cv.fit(X_train_scaled, y_train)

# Active features (non-zero coefficients)
active_mask = enet_cv.coef_ != 0
active_features = [f for f, m in zip(available_features, active_mask) if m]
print(f'ElasticNet: alpha={enet_cv.alpha_:.6f}, l1_ratio={enet_cv.l1_ratio_:.4f}')
print(f'Active features: {len(active_features)}/{len(available_features)}')

en_test_mae = mean_absolute_error(y_test, enet_cv.predict(X_test_scaled))
print(f'Test MAE: {en_test_mae:.4f}')

ElasticNet: alpha=0.005356, l1_ratio=0.9900
Active features: 36/68
Test MAE: 2.8015


In [61]:
# --- Fine-grained ElasticNet grid around winning hyperparameters ---
l1_fine = [0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 0.995, 1.0]
alphas_fine = np.sort(np.unique(np.concatenate([
    np.logspace(-5, -3, 30),
    np.linspace(0.001, 0.02, 40),
    np.logspace(-1.5, 1, 20),
])))

enet_fine = ElasticNetCV(
    l1_ratio=l1_fine, alphas=alphas_fine,
    cv=kf, max_iter=100000, tol=1e-5, n_jobs=-1,
)
enet_fine.fit(X_train_scaled, y_train)

en_fine_mae = mean_absolute_error(y_test, enet_fine.predict(X_test_scaled))
print(f'Fine ElasticNet: alpha={enet_fine.alpha_:.8f}, l1_ratio={enet_fine.l1_ratio_}')
print(f'Test MAE: {en_fine_mae:.4f}')

Fine ElasticNet: alpha=0.00733333, l1_ratio=1.0
Test MAE: 2.8021


## 5. Model B — Quantile Regression (median, L1)

**What Quantile Regression does**

Standard regression minimises the sum of *squared* errors (MSE). QR instead minimises a
weighted sum of *absolute* errors called the **pinball loss**:

$$L_\tau(y, \hat{y}) = \begin{cases} \tau \cdot (y - \hat{y}) & \text{if } y \geq \hat{y} \\ (1 - \tau) \cdot (\hat{y} - y) & \text{if } y < \hat{y} \end{cases}$$

At `quantile=0.5` (τ = 0.5), the two weights are equal, so the loss reduces to plain **MAE** —
the exact metric Kaggle evaluates. This is why QR is a natural fit here.

**Why the median minimises MAE**

The median is the value that minimises $\sum |y_i - \hat{y}|$. Squaring errors (MSE) pulls
predictions toward the *mean*, which is sensitive to outliers (e.g., shortened strike seasons
like 1994). The median is inherently robust to such extremes.

**The L1 regularisation (`alpha`)**

sklearn's `QuantileRegressor` adds an L1 penalty on the coefficients:

$$\min_\beta \; L_{0.5}(y, X\beta) + \alpha \cdot \|\beta\|_1$$

This simultaneously provides:
- **Shrinkage** — reduces coefficient magnitude to prevent overfitting
- **Feature selection** — drives uninformative coefficients to exactly zero (same as Lasso)

`alpha=0` is pure QR (no regularisation); higher `alpha` zeros out more features.
The `alpha` hyperparameter is tuned via `GridSearchCV` over 13 values (0 to 10).

**Why `solver='highs'`**

QR is a linear programming problem, not a differentiable optimisation. sklearn solves it
via the HiGHS LP solver, which is much faster than older interior-point methods for this
dataset size.

**How QR differs from Lasso**

Both use L1 regularisation so both can zero out coefficients, but they differ in loss function
and what they predict:

| | Lasso | QR (median, α>0) |
|---|---|---|
| **Loss function** | MSE (squared errors) | MAE (absolute errors) |
| **What it predicts** | Conditional **mean** | Conditional **median** |
| **Outlier sensitivity** | High — large residuals get squared | Low — all residuals weighted equally |
| **Regularisation** | L1 on coefficients | L1 on coefficients |
| **Optimisation** | Coordinate descent (differentiable) | Linear programming (non-differentiable) |

Lasso minimises $\sum(y_i - \hat{y}_i)^2 + \alpha\|\beta\|_1$, so a single outlier season can
pull all predictions toward it. QR minimises $\sum|y_i - \hat{y}_i| + \alpha\|\beta\|_1$,
treating all residuals equally regardless of magnitude. For a roughly symmetric target like
season wins, the mean and median are close — but QR is more stable when the distribution has
heavy tails or irregular seasons in the historical data.

**Why it complements ElasticNet**

| Property | ElasticNet | QR (median) |
|---|---|---|
| Loss function | MSE + L1+L2 | MAE + L1 |
| Sensitive to outliers | Yes (squared errors) | No (absolute errors) |
| Penalty type | L1 + L2 (elastic) | L1 only |

Because they make errors for *different reasons*, averaging them cancels independent noise —
which is why the 50/50 ensemble scored better than either model alone.

**Two variants are compared**:
- **QR on `active_features`**: uses only the features ElasticNet selected (non-zero coefficients).
  Fewer features reduces overfitting risk on the 1,812-row dataset.
- **QR on all features**: uses the full feature set, letting QR's own L1 penalty handle selection.
  The LP problem is larger but `alpha` can still zero out uninformative coefficients.

The variant with lower holdout MAE is chosen for the final ensemble.


In [62]:
qr_params = {'alpha': [0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]}
kf_qr = KFold(n_splits=5, shuffle=True, random_state=42)

# QR on ElasticNet-selected features
X_train_active = X_train_scaled[active_features]
X_test_active  = X_test_scaled[active_features]

qr_sel = GridSearchCV(
    QuantileRegressor(quantile=0.5, solver='highs'),
    qr_params, cv=kf_qr, scoring='neg_mean_absolute_error', n_jobs=-1,
)
qr_sel.fit(X_train_active, y_train)
qr_sel_mae = mean_absolute_error(y_test, qr_sel.predict(X_test_active))

# QR on all features
qr_all = GridSearchCV(
    QuantileRegressor(quantile=0.5, solver='highs'),
    qr_params, cv=kf_qr, scoring='neg_mean_absolute_error', n_jobs=-1,
)
qr_all.fit(X_train_scaled, y_train)
qr_all_mae = mean_absolute_error(y_test, qr_all.predict(X_test_scaled))

# Pick best variant
if qr_all_mae <= qr_sel_mae:
    best_qr_name, best_qr_mae, best_qr_model, best_qr_features = 'QR_all', qr_all_mae, qr_all, available_features
else:
    best_qr_name, best_qr_mae, best_qr_model, best_qr_features = 'QR_selected', qr_sel_mae, qr_sel, active_features

print(f'QR (selected {len(active_features)} feat): MAE={qr_sel_mae:.4f}, alpha={qr_sel.best_params_["alpha"]}')
print(f'QR (all {len(available_features)} feat):      MAE={qr_all_mae:.4f}, alpha={qr_all.best_params_["alpha"]}')
print(f'Best: {best_qr_name} (MAE={best_qr_mae:.4f})')

QR (selected 36 feat): MAE=2.8064, alpha=0.001
QR (all 68 feat):      MAE=2.8071, alpha=0.001
Best: QR_selected (MAE=2.8064)


## 6. Validation Summary

Reports both **holdout MAE** (80/20 split) and **10-fold CV MAE** (full training data) for
each model. The 10-fold estimate is more stable and uses all 1,812 rows, giving a better
sense of generalisation performance. Both metrics should be directionally consistent — if
they diverge, it may indicate overfitting to the particular 80/20 split.

In [63]:
kf10 = KFold(n_splits=10, shuffle=True, random_state=0)

en_cv = cross_val_score(
    ElasticNet(alpha=enet_fine.alpha_, l1_ratio=enet_fine.l1_ratio_, max_iter=100000, tol=1e-5),
    X_all_scaled, y, cv=kf10, scoring="neg_mean_absolute_error"
)

# Scale all features first with the fitted scaler, then subset to best_qr_features
X_all_qr_cv = X.copy()
X_all_qr_cv[other_cols] = scaler.transform(X[other_cols])
X_all_qr_cv = X_all_qr_cv[best_qr_features]

qr_cv = cross_val_score(
    QuantileRegressor(quantile=0.5, alpha=best_qr_model.best_params_["alpha"], solver="highs"),
    X_all_qr_cv, y, cv=kf10, scoring="neg_mean_absolute_error"
)

print(f"Model               Test MAE    10-fold CV MAE")
print(f"ElasticNet (fine)   {en_fine_mae:.4f}      {-en_cv.mean():.4f} +/- {en_cv.std():.4f}")
print(f"{best_qr_name:19s} {best_qr_mae:.4f}      {-qr_cv.mean():.4f} +/- {qr_cv.std():.4f}")

Model               Test MAE    10-fold CV MAE
ElasticNet (fine)   2.8021      2.7054 +/- 0.1596
QR_selected         2.8064      2.7104 +/- 0.1541


In [70]:
# --- 6B. Ablation Runner: baseline vs recommended feature sets ---
# Champion-challenger policy:
# - Baseline is always retained.
# - A candidate is promoted only if ensemble holdout MAE and ensemble CV MAE both improve.

from sklearn.model_selection import cross_val_predict

MIN_GAIN = 0.01
MIN_CV_GAIN = 0.005
CV_FOLDS = 5


def _safe_div(a, b, eps=1e-9):
    return a / (b + eps)


def _detect_year_col(df):
    for c in ["yearID", "year", "Year", "YEAR"]:
        if c in df.columns:
            return c
    return None


def _build_manager_features(managers_path):
    if not managers_path.exists():
        return pd.DataFrame(columns=["yearID", "teamID", "manager_tenure", "manager_change_flag"])

    mgr = pd.read_csv(managers_path)
    need_cols = {"playerID", "yearID", "teamID", "inseason"}
    if not need_cols.issubset(set(mgr.columns)):
        return pd.DataFrame(columns=["yearID", "teamID", "manager_tenure", "manager_change_flag"])

    # Keep one manager row per team-year (prefer inseason=0 where available).
    mgr = mgr.sort_values(["teamID", "yearID", "inseason"])
    mgr_year = mgr.groupby(["teamID", "yearID"], as_index=False).first()
    mgr_year = mgr_year.sort_values(["teamID", "yearID"]).reset_index(drop=True)

    tenure = []
    change = []
    prev_team = None
    prev_mgr = None
    current_tenure = 0

    for row in mgr_year.itertuples(index=False):
        team = row.teamID
        manager = row.playerID
        if team != prev_team:
            current_tenure = 1
            manager_changed = 0
        elif manager == prev_mgr:
            current_tenure += 1
            manager_changed = 0
        else:
            current_tenure = 1
            manager_changed = 1

        tenure.append(current_tenure)
        change.append(manager_changed)
        prev_team = team
        prev_mgr = manager

    mgr_year["manager_tenure"] = tenure
    mgr_year["manager_change_flag"] = change

    return mgr_year[["yearID", "teamID", "manager_tenure", "manager_change_flag"]]


def _build_teams_context_features(teams_path):
    cols_out = [
        "yearID", "teamID", "attendance_per_game", "park_factor_combo", "attendance_z"
    ]
    if not teams_path.exists():
        return pd.DataFrame(columns=cols_out)

    tm = pd.read_csv(teams_path)
    need_cols = {"yearID", "teamID", "attendance"}
    if not need_cols.issubset(set(tm.columns)):
        return pd.DataFrame(columns=cols_out)

    tm = tm.copy()
    if "Ghome" in tm.columns:
        tm["attendance_per_game"] = _safe_div(tm["attendance"].fillna(0), tm["Ghome"].replace(0, np.nan).fillna(0))
    elif "G" in tm.columns:
        tm["attendance_per_game"] = _safe_div(tm["attendance"].fillna(0), tm["G"].replace(0, np.nan).fillna(0))
    else:
        tm["attendance_per_game"] = tm["attendance"].fillna(0)

    # Park factor proxy centered around 100.
    if {"BPF", "PPF"}.issubset(set(tm.columns)):
        tm["park_factor_combo"] = (tm["BPF"].fillna(100) + tm["PPF"].fillna(100)) / 2.0
    elif "BPF" in tm.columns:
        tm["park_factor_combo"] = tm["BPF"].fillna(100)
    elif "PPF" in tm.columns:
        tm["park_factor_combo"] = tm["PPF"].fillna(100)
    else:
        tm["park_factor_combo"] = 100.0

    yr_mean = tm.groupby("yearID")["attendance_per_game"].transform("mean")
    yr_std = tm.groupby("yearID")["attendance_per_game"].transform("std")
    tm["attendance_z"] = _safe_div(tm["attendance_per_game"] - yr_mean, yr_std.replace(0, np.nan).fillna(0))

    return tm[cols_out].drop_duplicates(subset=["yearID", "teamID"])


def add_recommended_features(train_df, pred_df, cfg):
    tr = train_df.copy()
    pr = pred_df.copy()
    new_cols = []

    # Phase 2: Lahman manager stability features.
    if cfg.get("manager_stability", False):
        manager_features = _build_manager_features(DATA_DIR / "Managers.csv")
        if {"yearID", "teamID"}.issubset(tr.columns) and {"yearID", "teamID"}.issubset(pr.columns):
            tr = tr.merge(manager_features, on=["yearID", "teamID"], how="left")
            pr = pr.merge(manager_features, on=["yearID", "teamID"], how="left")
            tr["has_manager_info"] = tr["manager_tenure"].notna().astype(int)
            pr["has_manager_info"] = pr["manager_tenure"].notna().astype(int)
            tr[["manager_tenure", "manager_change_flag"]] = tr[["manager_tenure", "manager_change_flag"]].fillna(0)
            pr[["manager_tenure", "manager_change_flag"]] = pr[["manager_tenure", "manager_change_flag"]].fillna(0)
            new_cols += ["manager_tenure", "manager_change_flag", "has_manager_info"]

    # Phase 2b: Teams context (attendance + park factor).
    if cfg.get("teams_context", False):
        teams_features = _build_teams_context_features(DATA_DIR / "Teams.csv")
        if {"yearID", "teamID"}.issubset(tr.columns) and {"yearID", "teamID"}.issubset(pr.columns):
            tr = tr.merge(teams_features, on=["yearID", "teamID"], how="left")
            pr = pr.merge(teams_features, on=["yearID", "teamID"], how="left")
            tr["has_teams_context"] = tr["attendance_per_game"].notna().astype(int)
            pr["has_teams_context"] = pr["attendance_per_game"].notna().astype(int)
            fill_cols = ["attendance_per_game", "park_factor_combo", "attendance_z"]
            tr[fill_cols] = tr[fill_cols].fillna(0)
            pr[fill_cols] = pr[fill_cols].fillna(0)
            new_cols += fill_cols + ["has_teams_context"]

    if cfg.get("efficiency", False):
        for d in (tr, pr):
            d["run_conversion"] = _safe_div(d["R"] - d["HR"], d["H"] + d["BB"] - d["HR"])
            d["pitching_clutch"] = _safe_div(d["RA"], d["whip"])
            d["log_run_ratio"] = np.log1p(d["R"]) - np.log1p(d["RA"])
        new_cols += ["run_conversion", "pitching_clutch", "log_run_ratio"]

    if cfg.get("interactions", False):
        for d in (tr, pr):
            d["obp_x_slg"] = d["obp_proxy"] * d["slg_proxy"]
            d["run_diff_x_era"] = d["run_diff"] * d["era_vs_league"]
            d["iso_x_hr_rate"] = d["iso"] * d["hr_rate_off"]
        new_cols += ["obp_x_slg", "run_diff_x_era", "iso_x_hr_rate"]

    if cfg.get("era_z", False):
        year_col = _detect_year_col(tr)
        if year_col is not None and year_col in pr.columns:
            z_base = [c for c in [
                "R", "RA", "ERA", "HR", "BB", "SO", "AB", "H", "IPouts", "mlb_rpg"
            ] if c in tr.columns]
            if z_base:
                means = tr.groupby(year_col)[z_base].mean().add_suffix("_yr_mean").reset_index()
                stds = tr.groupby(year_col)[z_base].std().add_suffix("_yr_std").reset_index()

                tr = tr.merge(means, on=year_col, how="left").merge(stds, on=year_col, how="left")
                pr = pr.merge(means, on=year_col, how="left").merge(stds, on=year_col, how="left")

                z_cols = []
                for c in z_base:
                    m = f"{c}_yr_mean"
                    s = f"{c}_yr_std"
                    z = f"{c}_z_era"
                    tr[z] = _safe_div(tr[c] - tr[m], tr[s])
                    pr[z] = _safe_div(pr[c] - pr[m], pr[s])
                    z_cols.append(z)

                drop_cols = [f"{c}_yr_mean" for c in z_base] + [f"{c}_yr_std" for c in z_base]
                tr.drop(columns=drop_cols, inplace=True, errors="ignore")
                pr.drop(columns=drop_cols, inplace=True, errors="ignore")
                new_cols += z_cols

    tr = tr.replace([np.inf, -np.inf], 0).fillna(0)
    pr = pr.replace([np.inf, -np.inf], 0).fillna(0)
    return tr, pr, new_cols


def evaluate_config(name, cfg):
    tr_aug, pr_aug, added_cols = add_recommended_features(data_df, predict_df, cfg)

    feat_pool = base_features + added_cols
    feats = [c for c in feat_pool if c in tr_aug.columns and c in pr_aug.columns]

    X_local = tr_aug[feats]
    y_local = tr_aug["W"]

    X_tr, X_te, y_tr, y_te = train_test_split(X_local, y_local, test_size=0.2, random_state=42)

    oh = [c for c in X_tr.columns if c.startswith(("era_", "decade_"))]
    oth = [c for c in X_tr.columns if c not in oh]

    sc = StandardScaler()
    X_tr_s = X_tr.copy()
    X_te_s = X_te.copy()
    X_tr_s[oth] = sc.fit_transform(X_tr[oth])
    X_te_s[oth] = sc.transform(X_te[oth])

    kf_local = KFold(n_splits=5, shuffle=True, random_state=42)

    en_cv_local = ElasticNetCV(
        l1_ratio=[0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99],
        alphas=np.logspace(-4, 2, 60),
        cv=kf_local, max_iter=50000, tol=1e-4, n_jobs=-1,
    )
    en_cv_local.fit(X_tr_s, y_tr)

    active_mask_local = en_cv_local.coef_ != 0
    active_feats_local = [f for f, m in zip(feats, active_mask_local) if m]
    if not active_feats_local:
        active_feats_local = feats[:]

    l1_fine_local = [0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 0.995, 1.0]
    alphas_fine_local = np.sort(np.unique(np.concatenate([
        np.logspace(-5, -3, 30),
        np.linspace(0.001, 0.02, 40),
        np.logspace(-1.5, 1, 20),
    ])))

    en_fine_local = ElasticNetCV(
        l1_ratio=l1_fine_local, alphas=alphas_fine_local,
        cv=kf_local, max_iter=100000, tol=1e-5, n_jobs=-1,
    )
    en_fine_local.fit(X_tr_s, y_tr)
    en_mae_local = mean_absolute_error(y_te, en_fine_local.predict(X_te_s))

    qr_params_local = {"alpha": [0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]}

    X_tr_active = X_tr_s[active_feats_local]
    X_te_active = X_te_s[active_feats_local]

    qr_sel_local = GridSearchCV(
        QuantileRegressor(quantile=0.5, solver="highs"),
        qr_params_local, cv=kf_local, scoring="neg_mean_absolute_error", n_jobs=-1,
    )
    qr_sel_local.fit(X_tr_active, y_tr)
    qr_sel_mae_local = mean_absolute_error(y_te, qr_sel_local.predict(X_te_active))

    qr_all_local = GridSearchCV(
        QuantileRegressor(quantile=0.5, solver="highs"),
        qr_params_local, cv=kf_local, scoring="neg_mean_absolute_error", n_jobs=-1,
    )
    qr_all_local.fit(X_tr_s, y_tr)
    qr_all_mae_local = mean_absolute_error(y_te, qr_all_local.predict(X_te_s))

    if qr_all_mae_local <= qr_sel_mae_local:
        qr_name_local = "QR_all"
        qr_mae_local = qr_all_mae_local
        qr_model_local = qr_all_local
        qr_feats_local = feats
    else:
        qr_name_local = "QR_selected"
        qr_mae_local = qr_sel_mae_local
        qr_model_local = qr_sel_local
        qr_feats_local = active_feats_local

    en_holdout_preds = en_fine_local.predict(X_te_s).round().astype(int)
    qr_holdout_preds = qr_model_local.predict(X_te_s[qr_feats_local]).round().astype(int)
    ens_holdout_preds = ((en_holdout_preds + qr_holdout_preds) / 2).round().astype(int)
    ens_mae_local = mean_absolute_error(y_te, ens_holdout_preds)

    # Fixed-hyperparameter ensemble CV check to reduce false promotions.
    oh_all = [c for c in feats if c.startswith(("era_", "decade_"))]
    oth_all = [c for c in feats if c not in oh_all]
    sc_all = StandardScaler()
    X_all_en = X_local.copy()
    X_all_en[oth_all] = sc_all.fit_transform(X_local[oth_all])

    X_all_qr = X_local[qr_feats_local].copy()
    qr_oth_all = [c for c in qr_feats_local if c not in oh_all]
    sc_qr_all = StandardScaler()
    X_all_qr[qr_oth_all] = sc_qr_all.fit_transform(X_all_qr[qr_oth_all])

    kf_cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=0)
    en_cv_preds = cross_val_predict(
        ElasticNet(alpha=en_fine_local.alpha_, l1_ratio=en_fine_local.l1_ratio_, max_iter=100000, tol=1e-5),
        X_all_en,
        y_local,
        cv=kf_cv,
        n_jobs=-1,
    )
    qr_cv_preds = cross_val_predict(
        QuantileRegressor(quantile=0.5, alpha=qr_model_local.best_params_["alpha"], solver="highs"),
        X_all_qr,
        y_local,
        cv=kf_cv,
        n_jobs=-1,
    )
    en_cv_round = np.rint(en_cv_preds).astype(int)
    qr_cv_round = np.rint(qr_cv_preds).astype(int)
    ens_cv_preds = np.rint((en_cv_round + qr_cv_round) / 2).astype(int)
    ensemble_cv_mae_local = mean_absolute_error(y_local, ens_cv_preds)

    return {
        "name": name,
        "cfg": cfg,
        "data_df": tr_aug,
        "predict_df": pr_aug,
        "available_features": feats,
        "X": X_local,
        "y": y_local,
        "one_hot_cols": oh_all,
        "other_cols": oth_all,
        "enet_fine": en_fine_local,
        "en_fine_mae": en_mae_local,
        "best_qr_name": qr_name_local,
        "best_qr_mae": qr_mae_local,
        "best_qr_model": qr_model_local,
        "best_qr_features": qr_feats_local,
        "ensemble_mae": ens_mae_local,
        "ensemble_cv_mae": ensemble_cv_mae_local,
        "added_cols": added_cols,
    }


configs = [
    ("baseline", {}),
    ("manager_stability", {"manager_stability": True}),
    ("teams_context", {"teams_context": True}),
    ("manager_stability+teams_context", {"manager_stability": True, "teams_context": True}),
    ("era_z", {"era_z": True}),
    ("interactions", {"interactions": True}),
    ("efficiency", {"efficiency": True}),
    ("manager_stability+efficiency", {"manager_stability": True, "efficiency": True}),
    ("teams_context+efficiency", {"teams_context": True, "efficiency": True}),
    ("era_z+interactions", {"era_z": True, "interactions": True}),
]

results = []
for name, cfg in configs:
    print(f"\nRunning config: {name}")
    res = evaluate_config(name, cfg)
    results.append(res)
    print(
        f"  EN MAE={res['en_fine_mae']:.4f} | "
        f"{res['best_qr_name']} MAE={res['best_qr_mae']:.4f} | "
        f"Holdout Ensemble MAE={res['ensemble_mae']:.4f} | "
        f"CV Ensemble MAE={res['ensemble_cv_mae']:.4f}"
    )

baseline_result = next(r for r in results if r["name"] == "baseline")
best_result = min(results, key=lambda r: (r["ensemble_mae"], r["ensemble_cv_mae"]))
holdout_improvement = baseline_result["ensemble_mae"] - best_result["ensemble_mae"]
cv_improvement = baseline_result["ensemble_cv_mae"] - best_result["ensemble_cv_mae"]

if (
    best_result["name"] != "baseline"
    and holdout_improvement >= MIN_GAIN
    and cv_improvement >= MIN_CV_GAIN
):
    champion = best_result
    print(
        f"\nChampion promoted: {champion['name']} "
        f"(holdout +{holdout_improvement:.4f}, CV +{cv_improvement:.4f})"
    )
else:
    champion = baseline_result
    print(
        f"\nFallback to baseline: holdout improvement={holdout_improvement:.4f}, "
        f"CV improvement={cv_improvement:.4f}. Preserving original submission path."
    )

# Overwrite downstream variables so Sections 7+ stay unchanged but use champion config.
data_df = champion["data_df"]
predict_df = champion["predict_df"]
available_features = champion["available_features"]
X = champion["X"]
y = champion["y"]
one_hot_cols = champion["one_hot_cols"]
other_cols = champion["other_cols"]
enet_fine = champion["enet_fine"]
en_fine_mae = champion["en_fine_mae"]
best_qr_name = champion["best_qr_name"]
best_qr_mae = champion["best_qr_mae"]
best_qr_model = champion["best_qr_model"]
best_qr_features = champion["best_qr_features"]
champion_ensemble_mae = champion["ensemble_mae"]
champion_ensemble_cv_mae = champion["ensemble_cv_mae"]
champion_name = champion["name"]

ablation_summary = pd.DataFrame([
    {
        "config": r["name"],
        "en_mae": r["en_fine_mae"],
        "qr_name": r["best_qr_name"],
        "qr_mae": r["best_qr_mae"],
        "holdout_ensemble_mae": r["ensemble_mae"],
        "cv_ensemble_mae": r["ensemble_cv_mae"],
        "added_features": len(r["added_cols"]),
    }
    for r in results
]).sort_values(["holdout_ensemble_mae", "cv_ensemble_mae"])

print("\nAblation leaderboard (lower MAE is better):")
print(ablation_summary.to_string(index=False))


Running config: baseline
  EN MAE=2.8021 | QR_selected MAE=2.8064 | Holdout Ensemble MAE=2.7961 | CV Ensemble MAE=2.6992

Running config: manager_stability
  EN MAE=2.8028 | QR_selected MAE=2.8175 | Holdout Ensemble MAE=2.8017 | CV Ensemble MAE=2.7141

Running config: teams_context
  EN MAE=2.8052 | QR_selected MAE=2.8184 | Holdout Ensemble MAE=2.7879 | CV Ensemble MAE=2.6749

Running config: manager_stability+teams_context
  EN MAE=2.8051 | QR_all MAE=2.8190 | Holdout Ensemble MAE=2.7851 | CV Ensemble MAE=2.6860

Running config: era_z
  EN MAE=2.8268 | QR_selected MAE=2.8236 | Holdout Ensemble MAE=2.8540 | CV Ensemble MAE=2.6876

Running config: interactions
  EN MAE=2.8028 | QR_all MAE=2.8097 | Holdout Ensemble MAE=2.7961 | CV Ensemble MAE=2.7070

Running config: efficiency
  EN MAE=2.8036 | QR_selected MAE=2.8121 | Holdout Ensemble MAE=2.7824 | CV Ensemble MAE=2.7036

Running config: manager_stability+efficiency
  EN MAE=2.8044 | QR_selected MAE=2.8206 | Holdout Ensemble MAE=2.7769

## 7. Generate Submissions — Refit on ALL Training Data

For the final predictions, both models are **refitted on all 1,812 training rows** (not just
the 80% training split). A fresh `StandardScaler` is fit on the full training set to avoid
information from the holdout split leaking into the scaler statistics.

Each model produces a separate submission CSV with integer-rounded win predictions.
These individual files are inputs to the ensemble step in Section 8.

In [71]:
# --- Performance threshold check ---
mae_ok = champion_ensemble_mae < 2.80
if mae_ok:
    print(f'✓ Ensemble threshold passed — champion {champion_name} has holdout ensemble MAE < 2.80, submissions will be generated.')
else:
    print(f'⚠ Ensemble threshold NOT met — CSV files will be saved but NOT submitted.')
print(f'  Champion config: {champion_name}')
print(f'  Champion holdout ensemble MAE: {champion_ensemble_mae:.4f}')
print(f'  Champion CV ensemble MAE: {champion_ensemble_cv_mae:.4f}')
print(f'  ElasticNet (fine): {en_fine_mae:.4f}')
print(f'  Best QR ({best_qr_name}): {best_qr_mae:.4f}')

# --- Refit scaler + models on ALL training data ---
scaler_final = StandardScaler()
X_all_final = X.copy()
X_all_final[other_cols] = scaler_final.fit_transform(X[other_cols])

# Prepare predict set
predict_features = predict_df[available_features].copy()
predict_features[other_cols] = scaler_final.transform(predict_features[other_cols])

# --- ElasticNet submission ---
final_en = ElasticNet(
    alpha=enet_fine.alpha_, l1_ratio=enet_fine.l1_ratio_,
    max_iter=100000, tol=1e-5,
)
final_en.fit(X_all_final, y)
en_preds = final_en.predict(predict_features)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
en_file = SUBMISSIONS_DIR / f'submission_ElasticNet_{ts}.csv'
pd.DataFrame({'ID': predict_df['ID'], 'W': en_preds.round().astype(int)}).to_csv(en_file, index=False)
print(f'Saved: {en_file}')
print(f'  EN preds: mean={en_preds.mean():.1f}, std={en_preds.std():.1f}, min={en_preds.min():.1f}, max={en_preds.max():.1f}')

# --- QR submission ---
qr_feat_other = [c for c in best_qr_features if c not in one_hot_cols]

scaler_qr = StandardScaler()
X_all_qr = X[best_qr_features].copy()
X_all_qr[qr_feat_other] = scaler_qr.fit_transform(X_all_qr[qr_feat_other])

final_qr = QuantileRegressor(
    quantile=0.5, alpha=best_qr_model.best_params_['alpha'], solver='highs',
)
final_qr.fit(X_all_qr, y)

predict_qr = predict_df[best_qr_features].copy()
predict_qr[qr_feat_other] = scaler_qr.transform(predict_qr[qr_feat_other])
qr_preds = final_qr.predict(predict_qr)

qr_file = SUBMISSIONS_DIR / f'submission_{best_qr_name}_{ts}.csv'
pd.DataFrame({'ID': predict_df['ID'], 'W': qr_preds.round().astype(int)}).to_csv(qr_file, index=False)
print(f'Saved: {qr_file}')
print(f'  QR preds: mean={qr_preds.mean():.1f}, std={qr_preds.std():.1f}, min={qr_preds.min():.1f}, max={qr_preds.max():.1f}')


✓ Ensemble threshold passed — champion baseline has holdout ensemble MAE < 2.80, submissions will be generated.
  Champion config: baseline
  Champion holdout ensemble MAE: 2.7961
  Champion CV ensemble MAE: 2.6992
  ElasticNet (fine): 2.8021
  Best QR (QR_selected): 2.8064
Saved: /home/fredc/NTU-DSAI/kaggle-ha/submissions/submission_ElasticNet_20260402_070725.csv
  EN preds: mean=79.0, std=12.0, min=44.9, max=109.6
Saved: /home/fredc/NTU-DSAI/kaggle-ha/submissions/submission_QR_selected_20260402_070725.csv
  QR preds: mean=79.0, std=12.0, min=45.7, max=109.5


## 8. Ensemble — 50/50 Average (Best Public Score: 2.98765)

Simple averaging of two models with **different loss functions** (MSE-based ElasticNet vs
MAE-based QR) cancels independent prediction errors without adding overfitting risk.
No additional hyperparameters to tune.

The ensemble averages the **integer-rounded** individual predictions (matching the original
submission that scored 2.98765).

In [72]:
# 50-50 average of integer-rounded QR + ElasticNet predictions
en_rounded = en_preds.round().astype(int)
qr_rounded = qr_preds.round().astype(int)
avg_preds = (en_rounded + qr_rounded) / 2

avg_file = SUBMISSIONS_DIR / f'submission_avg_50-50_{ts}.csv'
pd.DataFrame({'ID': predict_df['ID'], 'W': avg_preds.round().astype(int)}).to_csv(avg_file, index=False)
print(f'Saved: {avg_file}')
print(f'  Avg preds: mean={avg_preds.mean():.1f}, std={avg_preds.std():.1f}')
print(f'  Disagreement (MAD): {np.abs(qr_rounded - en_rounded).mean():.2f}')

# Submit — only if the champion ensemble passed the MAE threshold
msg = f'Avg QR+ElasticNet 50-50 no franchise'
cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{avg_file}" -m "{msg}"'
if mae_ok:
    print(f'\nSubmitting champion {champion_name} with ensemble MAE={champion_ensemble_mae:.4f}')
    print(cmd)
    os.system(cmd)
else:
    print(
        f'\n⚠ Submission skipped: champion={champion_name}, '
        f'ensemble_mae={champion_ensemble_mae:.4f} (threshold 2.80)'
    )


Saved: /home/fredc/NTU-DSAI/kaggle-ha/submissions/submission_avg_50-50_20260402_070725.csv
  Avg preds: mean=79.0, std=12.0
  Disagreement (MAD): 0.22

Submitting champion baseline with ensemble MAE=2.7961
kaggle competitions submit -c sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics -f "/home/fredc/NTU-DSAI/kaggle-ha/submissions/submission_avg_50-50_20260402_070725.csv" -m "Avg QR+ElasticNet 50-50 no franchise"


100%|██████████| 3.34k/3.34k [00:00<00:00, 4.33kB/s]


Successfully submitted to SCTPDSAI-M3-DS2F-Coaching-MoneyBall Analytics

In [74]:
import time
# poll every 5 seconds for Kaggle to process, then run command.
for time in range(0, 60, 5):
    print(f'Waiting for Kaggle to process submission... ({time}s)')
    catch_output = os.popen(f'kaggle competitions submissions -c {COMPETITION_NAME} 2>/dev/null | head -n 5').read()
    if catch_output.strip() == '':
        print('No submission found. You may need to wait a bit longer for Kaggle to process the submission, then run the command again.')
    else:
        print(catch_output)
        break

Waiting for Kaggle to process submission... (0s)
fileName                                                date                        description                                              status                     publicScore  privateScore  
------------------------------------------------------  --------------------------  -------------------------------------------------------  -------------------------  -----------  ------------  
submission_avg_50-50_20260402_070725.csv                2026-04-01 23:07:29.657000  Avg QR+ElasticNet 50-50 no franchise                     SubmissionStatus.COMPLETE  2.98353                    
submission_avg_50-50_20260401_234207.csv                2026-04-01 15:42:10.577000  Avg QR+ElasticNet 50-50 no franchise                     SubmissionStatus.COMPLETE  2.98353                    
submission_avg_50-50_20260401_231303.csv                2026-04-01 15:22:01.303000  Avg QR+ElasticNet 50-50 no franchise                     SubmissionStatus.COMPLETE 